<h1 style=\"text-align: center; font-size: 50px;\"> 📦 Register Model </h1>

This notebook packages the **audio-native agentic workflow** as an **MLflow pyfunc model**, logs it with artifacts
(index, config), and registers it to the MLflow Model Registry for serving.

- Retrieval: **CLAP** audio embeddings over timestamped windows (+ **MMR** reranker)
- Generation: **Qwen Omni** listens to the selected audio windows and answers (no transcripts required)
- Orchestration: **LangGraph** (relevance → memory → retrieve → rerank → answer → memoize)
- Vector store: **FAISS** (in-model artifact or built on first run)
- Memory: disk-backed key-value cache (per-corpus+question)


# Notebook Overview

- Start Execution
- Define User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- KV Memory
- LLM Setup
- State Model
- Node Functions
- Graph Definition
- Graph Visualization
- Generated Answer
- Message History

# Start Execution

In [1]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations
from pathlib import Path  # Object-oriented file system paths

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.utils import (  # Utility functions for logging, LLM I/O, and schema generation
    load_config,
    load_secrets,
    load_secrets_to_env,
    get_project_root,
    logger,
    setup_model_environment,
)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 116 ms, sys: 67.6 ms, total: 184 ms
Wall time: 4.34 s


In [4]:
from __future__ import annotations  # Enables postponed evaluation of annotations (PEP 563)

# ─────── Standard Library ───────
import json  # JSON serialization and deserialization
import warnings  # Issue warning messages
from collections import namedtuple  # Factory for creating tuple subclasses with named fields
from datetime import datetime  # Date and time utilities
from pathlib import Path  # Object-oriented filesystem paths
from typing import Any, Dict, List, Literal, Optional, TypedDict  # Type hinting support
import numpy as np  # Numerical operations and array handling
import soundfile as sf  # Reading and writing sound files

# ─────── Third-Party Packages ───────
import mlflow  # Model tracking and serving framework
import mlflow.pyfunc  # MLflow Python function interface for custom models
from mlflow.tracking import MlflowClient  # Interface to interact with MLflow tracking server for experiments, runs, and artifacts
from IPython.display import Markdown, display  # IPython utilities for notebook output formatting
from tqdm import tqdm  # Visual progress bar for loops
import torch # PyTorch for tensor computations and deep learning
import torchaudio # Audio processing library built on PyTorch
import faiss # Library for efficient similarity search and clustering of dense vectors
import pandas as pd # Data manipulation and analysis

# Qwen Omni (audio+video+text) – both full & Thinker-only variants
from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration # Qwen Omni processor and model
from transformers import AutoProcessor as ClapProcessor, ClapModel # CLAP processor and model for audio embeddings
from qwen_omni_utils import process_mm_info     # official utils to prep audio/video inputs

# ─────── LangChain Core & Community ───────
from langgraph.graph import StateGraph, END

# ─────── Local application-specific imports ───────
from src.agentic_workflow import build_audio_agentic_graph # Custom workflow builder for agentic tasks
from src.model_selection import ModelSelector
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state
from src.simple_kv_memory import _mem_get, _mem_put  # Functions for getting and putting items in memory
from src.generate_test_audio import generate_test_audio, generate_and_convert_formats  # Functions to generate test audio files
from src.segment_audio_embeddings import (  # Functions for segmenting audio and extracting embeddings
    clap_embed_audio,
    clap_embed_text,
    segment_audio_embeddings, 
    rerank_hits_mmr, 
    retrieve_and_rerank,
    ensure_wav
)
from src.agentic_audio_rag_model import AudioAgenticPyFunc
from core.agentic_audio_rag_service.agentic_audio_rag_service import AgenticAudioService




/opt/conda/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


# Configure Settings

In [5]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [6]:
project_root = get_project_root()
MEMORY: SimpleKVMemory

QUESTION: str = "What is the main idea of the content?"
DOCS: list
FILE_ID: str

INPUT_PATH: Path = Path("../data/input/meeting_recording")
MEMORY_PATH: Path = Path("../data/memory")
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"
DATA_PATH = "../data/input/meeting_recording"
DEMO_FOLDER = "../demo"

# LLAMA_MODEL_PATH = "/home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"
SAMPLE_MEDIA_PATH = INPUT_PATH / "sample_tts.mp3"
CONTEXT_WINDOW = 8192
MAX_TOKENS = CONTEXT_WINDOW // 8
CHUNK_SIZE = CONTEXT_WINDOW // 2
CHUNK_OVERLAP = CHUNK_SIZE // 8  

MLFLOW_EXPERIMENT_NAME = "AIStudio-Agentic-Audio-RAG-Experiment"
MLFLOW_RUN_NAME = "AIStudio-Agentic-Audio-RAG-Run"
MLFLOW_MODEL_NAME = "AIStudio-Agentic-Audio-RAG-Model"

# --- Retrieval / Rerank params (must match run-workflow) ---
MEMORY_FILENAME = "kv_memory.jsonl"
INDEX_VECS_NPY = "audio_vecs.npy"
INDEX_META_JSON = "audio_meta.json"
RELEVANCE_THRESHOLD = 0.18
FETCH_K = 24     # breadth for stage-1
TOP_K   = 6      # final segments


# Configuration parameter
MODEL_SOURCE = "hugging-face-cloud"  # Can be: "local", "hugging-face-local", "hugging-face-cloud"


In [7]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

✅ Loaded 1 secrets into environment variables.
✅ Configuration loaded successfully
✅ Secrets loaded successfully


In [8]:
logger.info('Notebook execution started.')

## Verify Assets

In [9]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

def log_secrets_status(secrets: Dict[str, Any], success_message: str, failure_message: str) -> None:
    """
    Logs the status of secrets based on their existence.

    Parameters:
        secrets (Dict[str, Any]): Secrets retrieved to check if they exist.
        success_message (str): Message to log if secrets exists.
        failure_message (str): Message to log if secrets do not exist.
    """
    if secrets:
        logger.info(f"Project secrets are available. {success_message}")
    else:
        logger.info(f"There are no project secrets found. {failure_message}")

In [10]:
log_asset_status(
    asset_path=INPUT_PATH,
    asset_name="Input Data",
)

log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="Config",
)

log_secrets_status(
    secrets=secrets,
    success_message="",
    failure_message="Please check if the secrets were propely connfigured in your secrets yaml file or in Secrets Manager."
)

# KV Memory

In [11]:
memory: SimpleKVMemory = SimpleKVMemory(MEMORY_PATH)
memory.set('dummy key', 'dummy value')

# Prepare Audio Files for Inference

In [12]:
logger.info("🎧 Scanning directory for media files: %s", INPUT_PATH)

# Prefer an audio-native LLM
AUDIO_LLM = {
 #   "MiDaSheng": ("MiSpeech/MiDaShengLM-7B-GGUF"),
 #   "Kimi": ("Moonshot-AI/Kimi-Audio-7B-Instruct"),
    "Qwen": ("Qwen/Qwen2.5-Omni-7B"), 
}["Qwen"]

# Supported media types
AUDIO_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv"}
MEDIA_EXTS = AUDIO_EXTS | VIDEO_EXTS

SRC_DIR   = project_root / "src"
DATA_DIR  = project_root / "data"
INPUT_DIR = DATA_DIR / "input"     # media to index (same as run-workflow)
ARTIF_DIR = project_root / "artifacts" # temp artifacts for logging

# Make src importable
sys.path.insert(0, str(SRC_DIR))

print("Project root:", project_root)
print("Src dir    :", SRC_DIR)
print("Data dir   :", DATA_DIR)
print("Inputs     :", INPUT_DIR)
print("Artifacts  :", ARTIF_DIR)

# Ensure HF cache paths live in the project area (matches README/setup)
setup_model_environment()

Project root: /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph
Src dir    : /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/src
Data dir   : /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/data
Inputs     : /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/data/input
Artifacts  : /home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/artifacts


In [13]:
# Reuse the CLAP init + embedding utilities from your run-workflow notebook
# If you already defined them earlier in this kernel, skip redefining.
from transformers import ClapProcessor, ClapModel
import torch

# CLAP init
CLAP_REPO = "laion/clap-htsat-unfused"
clap_device = "cuda" if torch.cuda.is_available() else "cpu"
clap_processor = ClapProcessor.from_pretrained(CLAP_REPO)
clap_model = ClapModel.from_pretrained(CLAP_REPO).to(clap_device).eval()

try:
    clap_model.to("cpu")
    clap_device = "cpu"
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("CLAP moved to CPU; GPU cache cleared")
except Exception as e:
    print("Skipping CLAP offload:", e)

def _resample_to_48k(wav: np.ndarray, sr: int, target_sr: int = 48000) -> np.ndarray:
    if sr == target_sr:
        return wav.astype(np.float32, copy=False)
    try:
        import torchaudio
        t = torch.as_tensor(wav, dtype=torch.float32).unsqueeze(0)
        t48 = torchaudio.functional.resample(t, sr, target_sr)
        return t48.squeeze(0).cpu().numpy().astype(np.float32)
    except Exception:
        x = np.linspace(0, 1, num=wav.shape[0], dtype=np.float64, endpoint=False)
        y = np.interp(np.linspace(0, 1, num=int(round(wav.shape[0] * target_sr / sr)), endpoint=False),
                      x, wav.astype(np.float64, copy=False))
        return y.astype(np.float32)

@torch.no_grad()
def clap_embed_audio(wav: np.ndarray, sr: int) -> np.ndarray:
    wav48 = _resample_to_48k(wav, sr, 48000)
    inp = clap_processor(audios=[wav48], sampling_rate=48000, return_tensors="pt").to(clap_device)
    out = clap_model.get_audio_features(**inp)
    vec = out.cpu().numpy()[0]
    vec = vec / (np.linalg.norm(vec) + 1e-12)
    return vec.astype(np.float32)

@torch.no_grad()
def clap_embed_text(query: str) -> np.ndarray:
    inp = clap_processor(text=[query], return_tensors="pt").to(clap_device)
    out = clap_model.get_text_features(**inp)
    vec = out.cpu().numpy()[0]
    vec = vec / (np.linalg.norm(vec) + 1e-12)
    return vec.astype(np.float32)

# Segmentation (same as run-workflow)
from typing import List, Tuple, Dict, Any
def segment_audio(wav_path: str, window_s: float = 30.0, hop_s: float = 15.0) -> List[Tuple[int, int, np.ndarray, int]]:
    audio, sr = sf.read(wav_path)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if audio.dtype != np.float32:
        audio = audio.astype(np.float32)
    n = len(audio); win = int(window_s * sr); hop = int(hop_s * sr)
    if n == 0: return []
    segs, i = [], 0
    while i < n:
        j = min(i + win, n)
        segs.append((i, j, audio[i:j], sr))
        if j == n: break
        i += hop
    return segs

# In-memory FAISS index shell
class AudioIndex:
    def __init__(self, dim: int = 512):
        self.index = faiss.IndexFlatIP(dim)
        self.meta: List[Dict[str, Any]] = []
    def add(self, vecs: np.ndarray, metas: List[Dict[str, Any]]):
        # cosine via normalized IP
        vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
        self.index.add(vecs.astype(np.float32))
        self.meta.extend(metas)
    def search(self, qvec: np.ndarray, k: int = 6) -> List[Dict[str, Any]]:
        qvec = qvec.astype(np.float32)
        qvec = qvec / (np.linalg.norm(qvec) + 1e-12)
        D, I = self.index.search(qvec[np.newaxis, :], k)
        out = []
        for idx, score in zip(I[0], D[0]):
            if 0 <= idx < len(self.meta):
                m = dict(self.meta[idx]); m["score"] = float(score)
                out.append(m)
        return out

# Build index from INPUT_DIR and snapshot to artifacts/
AUDIO_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}
MEDIA_EXTS = AUDIO_EXTS | VIDEO_EXTS

ARTIF_DIR.mkdir(parents=True, exist_ok=True)
(ARTIF_DIR / "index").mkdir(parents=True, exist_ok=True)
(ARTIF_DIR / "config").mkdir(parents=True, exist_ok=True)
(ARTIF_DIR / "memory").mkdir(parents=True, exist_ok=True)  # empty initial memory

# Collect media
media_paths = []
for p in sorted(Path(INPUT_DIR).rglob("*")):
    if any(part.startswith(".") and part not in {".", ".."} for part in p.parts):
        continue
    if p.is_file() and p.suffix.lower() in MEDIA_EXTS:
        media_paths.append(p)

# Embed segments
audio_index = AudioIndex(dim=512)
for media_path in media_paths:
    wav_path = ensure_wav(AUDIO_EXTS, VIDEO_EXTS, str(media_path))
    segs = segment_audio(wav_path, window_s=30.0, hop_s=15.0)
    if not segs: continue
    vecs, metas = [], []
    for (s0, s1, wav_seg, sr) in segs:
        v = clap_embed_audio(wav_seg, sr); vecs.append(v)
        metas.append({
            "file_path": str(media_path),
            "file_name": media_path.name,
            "wav_path": wav_path,
            "start_s": float(s0 / sr),
            "end_s": float(s1 / sr),
        })
    audio_index.add(np.stack(vecs, axis=0), metas)

# Persist index vectors + metadata as model artifacts
# We need the raw (already normalized) vectors; FAISS can't be pickled easily across runtimes.
# Re-run a pass to collect vectors in the same order FAISS used:
# (For simplicity, we re-embed here; for large corpora, persist as you add)
vecs = []
for m in audio_index.meta:
    audio, sr = sf.read(m["wav_path"])
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    i0 = int(m["start_s"] * sr); i1 = int(m["end_s"] * sr)
    wav_seg = audio[i0:i1].astype(np.float32, copy=False)
    vecs.append(clap_embed_audio(wav_seg, sr))
vecs = np.stack(vecs, axis=0).astype(np.float32)
np.save(ARTIF_DIR / "index" / INDEX_VECS_NPY, vecs)

with open(ARTIF_DIR / "index" / INDEX_META_JSON, "w") as f:
    json.dump(audio_index.meta, f, ensure_ascii=False, indent=2)

# Write a simple runtime config
config = {
    "relevance_threshold": RELEVANCE_THRESHOLD,
    "fetch_k": FETCH_K,
    "top_k": TOP_K,
    "clap_repo": CLAP_REPO,
    "media_root": str(INPUT_DIR),
}
with open(ARTIF_DIR / "config" / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Indexed segments:", len(audio_index.meta))


CLAP moved to CPU; GPU cache cleared
Indexed segments: 90


# MLflow Registration

In [14]:
# %%time

from packaging.version import parse as vparse

# mlflow.set_tracking_uri(MLFLOW_TRACKING)
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {MLFLOW_EXPERIMENT_NAME}")

MEMORY_PATH: Path = Path("../data/memory")

# === Get model path from config ===
model_path = config.get("model_path")
if model_path and os.path.exists(model_path):
    logger.info(f"✅ Model file found at: {model_path}")
else:
    logger.info(f"⚠️ Warning: Model file not found at {model_path}. Please verify the path in config.yaml.")

logger.info(f'Starting the experiment: {MLFLOW_EXPERIMENT_NAME}')
logger.info(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")

# # Read requirements from your repo (or pin a minimal list inline)
# req_path = project_root / "requirements.txt"
# if req_path.exists():
#     with open(req_path, "r") as f:
#         pip_reqs = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]
# else:
#     pip_reqs = [
#         "mlflow>=3.1.0",
#         "langgraph>=0.2.0",
#         "transformers>=4.41.0",
#         "torch>=2.1.0",
#         "faiss-cpu>=1.7.4",
#         "soundfile>=0.12.1",
#         "huggingface_hub>=0.23.0",
#         "tabulate>=0.9.0",
#     ]

with mlflow.start_run(run_name=f"register-{MLFLOW_RUN_NAME}") as run:
    # Print the artifact URI for reference
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Artifacts mapping for pyfunc
    artifacts = {
        "index_dir": str(ARTIF_DIR / "index"),
        "config_path": str(ARTIF_DIR / "config" / "config.json"),
        "memory_dir": str(ARTIF_DIR / "memory"),
    }

    # Include local src so the server can import utils, model_selection, etc.
    code_paths = [str(SRC_DIR)]

    # model_info = mlflow.pyfunc.log_model(
    #     artifact_path="model",
    #     python_model=AudioAgenticPyFunc(),
    #     artifacts=artifacts,
    #     code_paths=code_paths,
    #     pip_requirements=pip_reqs,
    #     registered_model_name=MODEL_NAME,
    # )
    
    #  # Log model artifacts using custom ChatbotService
    # ChatbotService.log_model(
    #     artifact_path=MLFLOW_MODEL_NAME,
    #     config_path=CONFIG_PATH,
    #     docs_path=DATA_PATH,
    #     secrets_dict=secrets if secrets else None,
    #     model_path=model_path,
    #     demo_folder=DEMO_FOLDER
    # )
    
     # Log model artifacts using custom ChatbotService
    AgenticAudioService.log_model(
        artifact_path=MLFLOW_MODEL_NAME,
        config_path=CONFIG_PATH,
        docs_path=DATA_PATH,
        secrets_dict=secrets if secrets else None,
        model_path=model_path,
        demo_folder=DEMO_FOLDER
    )

    # log_kwargs = dict(
    #     artifact_path=MLFLOW_MODEL_NAME,
    #     python_model=AudioAgenticPyFunc(),  # now imported from src file (no notebook pickling)
    #     artifacts=artifacts,
    #     pip_requirements=pip_reqs,
    #     registered_model_name=MLFLOW_MODEL_NAME,
    # )
    # if vparse(mlflow.__version__) >= vparse("3.0.0"):
    #     log_kwargs["code_paths"] = code_paths
    # else:
    #     log_kwargs["code_path"] = code_paths  # backward compat
    
    # model_info = mlflow.pyfunc.log_model(**log_kwargs)
    
    # Construct the URI for the logged model
    model_uri = f"runs:/{run.info.run_id}/{MLFLOW_MODEL_NAME}"

    # Register the model into MLflow Model Registry
    mlflow.register_model(
        model_uri=model_uri,
        name=MLFLOW_MODEL_NAME
    )

print("Logged model at:", model_uri)
print("Registered name:", MLFLOW_MODEL_NAME)
logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

#######################################################################################


2025/08/26 00:48:58 INFO mlflow.tracking.fluent: Experiment with name 'AIStudio-Agentic-Audio-RAG-Experiment' does not exist. Creating a new experiment.


Using MLflow tracking URI: /phoenix/mlflow
Experiment: AIStudio-Agentic-Audio-RAG-Experiment


2025/08/26 00:49:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'AIStudio-Agentic-Audio-RAG-Model'.
2025/08/26 00:49:11 WARNING mlflow.tracking._model_registry.fluent: Run with id a535925ae5ef402fbfda2ce6c820eca6 has no artifacts at artifact path 'AIStudio-Agentic-Audio-RAG-Model', registering model based on models:/m-5896e5459f8d44d188401ef6b54c24fb instead
Created version '1' of model 'AIStudio-Agentic-Audio-RAG-Model'.


Logged model at: runs:/a535925ae5ef402fbfda2ce6c820eca6/AIStudio-Agentic-Audio-RAG-Model
Registered name: AIStudio-Agentic-Audio-RAG-Model


In [23]:
loaded = mlflow.pyfunc.load_model(model_info.model_uri)

TEST_Q = "What is the main idea of the content?"
payload = [{"question": TEST_Q, "file_id": "global"}]

res = loaded.predict(payload)
print(json.dumps(res, indent=2)[:1200], "...")


Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 24.00 GiB of which 0 bytes is free. Of the allocated memory 22.91 GiB is allocated by PyTorch, and 249.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Message History

In [21]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).